In [1]:
!pip install -q ultralytics opencv-python-headless pandas openpyxl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.9 MB/s eta 0:00:00


In [2]:
!unzip Data.zip

Archive:  Data.zip
  inflating: classes.txt             
   creating: images/
  inflating: images/003b377c-Bakso_24.jpeg  
  inflating: images/040444f9-Opor_ayam_5.jpeg  
  inflating: images/041adefc-Ayam_goreng_4.jpeg  
  inflating: images/047ad7db-Telur_ceplok_15.jpeg  
  inflating: images/04f0a539-Ayam_goreng_11.jpeg  
  inflating: images/05d9c981-Bakso_13.jpeg  
  inflating: images/0640b228-Rendang_1.jpg  
  inflating: images/0704c1de-Bakso_18.jpeg  
  inflating: images/0727580c-Nasi_pecel_12.jpeg  
  inflating: images/08266f01-Nasi_padang_25.jpeg  
  inflating: images/0872e1c3-Nasi_padang_15.jpeg  
  inflating: images/089cfcb3-Opor_ayam_14.jpeg  
  inflating: images/08c8f22e-Pempek_lenjer_8.jpeg  
  inflating: images/08e0132b-Tahu_goreng_14.jpeg  
  inflating: images/0a50df11-Opor_ayam_15.jpeg  
  inflating: images/0a975758-Tempe_goreng_16.jpeg  
  inflating: images/0b55c11e-Opor_ayam_10.jpeg  
  inflating: images/0b5a0bb3-Nasi_putih_25.jpeg  
  inflating: images/0be7f2c4-Ayam_gor

In [3]:
import os
from pathlib import Path
import pandas as pd
import shutil
import json
import math
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import cv2
from google.colab import files

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
!pip install -U ultralytics

In [5]:
ROOT = Path('/content')
IMG_DIR = ROOT / 'images'
LBL_DIR = ROOT / 'labels'
CLASSES_TXT = ROOT / 'classes.txt'
XLSX_PATH = ROOT / 'Kandungan_nutrisi.xlsx'
DATASET_DIR = ROOT / 'dataset'
TRAIN_IMG = DATASET_DIR / 'images' / 'train'
VAL_IMG = DATASET_DIR / 'images' / 'val'
TRAIN_LBL = DATASET_DIR / 'labels' / 'train'
VAL_LBL = DATASET_DIR / 'labels' / 'val'


print('Checking files...')
print('Images exists:', IMG_DIR.exists())
print('Labels exists:', LBL_DIR.exists())
print('classes.txt exists:', CLASSES_TXT.exists())
print('xlsx exists:', XLSX_PATH.exists())

Checking files...
Images exists: True
Labels exists: True
classes.txt exists: True
xlsx exists: True


In [6]:
if not CLASSES_TXT.exists():
  raise FileNotFoundError(f"{CLASSES_TXT} not found. Please upload classes.txt to /content/classes.txt")


with open(CLASSES_TXT, 'r', encoding='utf-8') as f:
  classes = [x.strip() for x in f.read().strip().splitlines() if x.strip()]


print('Classes found ({}):'.format(len(classes)), classes)


if not XLSX_PATH.exists():
  raise FileNotFoundError(f"{XLSX_PATH} not found. Please upload the Excel file to /content/Kandungan_nutrisi.xlsx")


nutrition_df = pd.read_excel(XLSX_PATH)
if 'class' in map(str.lower, nutrition_df.columns):
  cols = list(nutrition_df.columns)
  class_col = next(c for c in cols if c.lower() == 'class')
  nutrition_df = nutrition_df.rename(columns={class_col: 'class'})
elif 'name' in map(str.lower, nutrition_df.columns):
  cols = list(nutrition_df.columns)
  class_col = next(c for c in cols if c.lower() == 'name')
  nutrition_df = nutrition_df.rename(columns={class_col: 'class'})
else:
  nutrition_df = nutrition_df.rename(columns={nutrition_df.columns[0]: 'class'})


nutrition_df['class'] = nutrition_df['class'].astype(str).str.strip()
nutrition_df = nutrition_df.set_index('class')
print('Nutrition table columns:', nutrition_df.columns.tolist())
print('Nutrition table preview:')
print(nutrition_df.head())

Classes found (29): ['Ayam goreng', 'Bakso besar', 'Bakso sedang', 'Gudeg', 'Martabak manis', 'Martabak telur', 'Mie ayam', 'Nasi goreng', 'Nasi padang', 'Nasi pecel', 'Nasi putih', 'Opor ayam', 'Pangsit', 'Pangsit goreng', 'Pempek adaan', 'Pempek kapal selam', 'Pempek kulit', 'Pempek lenjer', 'Rawon', 'Rendang', 'Sate ayam', 'Sate ayam 100g', 'Soto', 'Tahu goreng', 'Tahu telor', 'Telur ceplok', 'Telur dadar', 'Telur rebus', 'Tempe goreng']
Nutrition table columns: ['Kalori', 'Lemak', 'Karbohidrat', 'Protein', 'Serat', 'Gula', 'Ukuran']
Nutrition table preview:
                Kalori  Lemak  Karbohidrat  Protein  Serat   Gula    Ukuran
class                                                                      
Ayam goreng        260  14.55        10.76    21.93    1.4   0.26      100g
Bakso besar         85   5.53         3.18     5.21    0.2   0.64    1 buah
Bakso sedang        57   3.69         2.12     3.47    0.1   0.42    1 buah
Gudeg              306  11.34        54.32     4.37 

In [7]:
os.makedirs(TRAIN_IMG, exist_ok=True)
os.makedirs(VAL_IMG, exist_ok=True)
os.makedirs(TRAIN_LBL, exist_ok=True)
os.makedirs(VAL_LBL, exist_ok=True)


img_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
all_images = [p for p in IMG_DIR.iterdir() if p.suffix.lower() in img_exts]
print('Found', len(all_images), 'images in', IMG_DIR)


pairs = []
for img_path in all_images:
  label_path = LBL_DIR / (img_path.stem + '.txt')
  if label_path.exists():
    pairs.append((img_path, label_path))
  else:
    print('Warning: label not found for image', img_path.name, '-> skipping')


if len(pairs) == 0:
  raise RuntimeError('No labeled image found. Please ensure labels are present in /content/labels')


img_paths = [str(p[0]) for p in pairs]
train_imgs, val_imgs = train_test_split(img_paths, test_size=0.2, random_state=42)

for p in train_imgs:
  img = Path(p)
  shutil.copy(img, TRAIN_IMG / img.name)
  shutil.copy(LBL_DIR / (img.stem + '.txt'), TRAIN_LBL / (img.stem + '.txt'))
for p in val_imgs:
  img = Path(p)
  shutil.copy(img, VAL_IMG / img.name)
  shutil.copy(LBL_DIR / (img.stem + '.txt'), VAL_LBL / (img.stem + '.txt'))


print('Train images:', len(list(TRAIN_IMG.iterdir())))
print('Val images:', len(list(VAL_IMG.iterdir())))

data_yaml = {
  'path': str(DATASET_DIR),
  'train': 'images/train',
  'val': 'images/val',
  'nc': len(classes),
  'names': classes
}


with open(DATASET_DIR / 'data.yaml', 'w', encoding='utf-8') as f:
  f.write(json.dumps(data_yaml, ensure_ascii=False, indent=2))

print('Wrote', DATASET_DIR / 'data.yaml')

Found 365 images in /content/images
Train images: 292
Val images: 73
Wrote /content/dataset/data.yaml


In [9]:
model = YOLO("yolov8s.pt")

model.train(
    data="/content/dataset/data.yaml",
    epochs=150,
    imgsz=640,
    batch=8,
    freeze=10,
    lr0=0.003,
    lrf=0.1,
    mosaic=1.0,          # ON (penting)
    mixup=0.2,           # bantu generalisasi
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    fliplr=0.5,
    flipud=0.1,

    # Jangan terlalu keras
    degrees=0.0,
    shear=0.0,
    perspective=0.0
)

Ultralytics 8.3.237 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ce3e13c2e10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.04104

In [ ]:
model = YOLO("runs/detect/train2/weights/best.pt")

model.train(
    data="/content/dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    freeze=0,
    lr0=0.001,
    lrf=0.01
)

Ultralytics 8.3.237 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/train2/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Ayam goreng', 1: 'Bakso besar', 2: 'Bakso sedang', 3: 'Gudeg', 4: 'Martabak manis', 5: 'Martabak telur', 6: 'Mie ayam', 7: 'Nasi goreng', 8: 'Nasi padang', 9: 'Nasi pecel', 10: 'Nasi putih', 11: 'Opor ayam', 12: 'Pangsit', 13: 'Pangsit goreng', 14: 'Pempek adaan', 15: 'Pempek kapal selam', 16: 'Pempek kulit', 17: 'Pempek lenjer', 18: 'Rawon', 19: 'Rendang', 20: 'Sate ayam', 21: 'Sate ayam 100g', 22: 'Soto', 23: 'Tahu goreng', 24: 'Tahu telor', 25: 'Telur ceplok', 26: 'Telur dadar', 27: 'Telur rebus', 28: 'Tempe goreng'}
 obb: None
 orig_img: array([[[119, 146, 172],
         [121, 148, 174],
         [122, 152, 177],
         ...,
         [161, 171, 188],
         [160, 170, 187],
         [160, 170, 187]],
 
        [[120, 147, 173],
         [122, 149, 175],
         [123, 153, 178],
         ...,
         [161, 171, 188],
         

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

trained_weights = None
weights_dir = Path('runs')
for p in weights_dir.rglob('best.pt'):
  if 'nutriscan_run1' in str(p):
    trained_weights = str(p)
  break

if trained_weights is None:
  candidates = list(weights_dir.rglob('*.pt'))
  if candidates:
    trained_weights = str(sorted(candidates, key=lambda x: x.stat().st_mtime)[-1])

print('Using weights:', trained_weights)
infer_model = YOLO(trained_weights or 'yolov8n.pt')


import numpy as np


def detect_and_compute(image_path, default_weights_g=100):
  img = cv2.imread(str(image_path))[:, :, ::-1]
  results = infer_model.predict(source=str(image_path), imgsz=640, conf=0.25, save=False)
  res = results[0]
  annotated = res.boxes.plot(show=False)

  detections = []
  for box in res.boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    xyxy = box.xyxy[0].tolist()
    cls_name = classes[cls_id]

    detections.append({'class_id': cls_id, 'class_name': cls_name, 'conf': conf, 'xyxy': xyxy})

  mapping = {}
  for d in detections:
    mapping.setdefault(d['class_name'], default_weights_g)


  nutrition_results = []
  for d in detections:
    cname = d['class_name']
    weight = mapping.get(cname, default_weights_g)

    if cname in nutrition_df.index:
      per100 = nutrition_df.loc[cname]
      factor = weight / 100.0
      estimated = (per100 * factor).to_dict()
    else:
      estimated = {col: None for col in nutrition_df.columns}
    nutrition_results.append({'detection': d, 'weight_g': weight, 'nutrition': estimated})


  out_path = Path('/content') / f'annotated_{Path(image_path).name}'
  cv2.imwrite(str(out_path), annotated[:, :, ::-1])


  return nutrition_results, str(out_path)

Using weights: None


NameError: name 'YOLO' is not defined

In [ ]:
import zipfile

best_pt = trained_weights or None
if best_pt:
  zipname = '/content/nutriscan_model.zip'
  with zipfile.ZipFile(zipname, 'w') as z:
    z.write(best_pt, arcname=Path(best_pt).name)
  print('Model zipped to', zipname)
else:
  print('No trained model found to package.')

Model zipped to /content/nutriscan_model.zip


In [ ]:
files.download("/content/nutriscan_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>